In [93]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [94]:
input_data = "input_files/mathematical_reasoning_0117.csv"
output_data = "output_files/mathematical_reasoning_0117.xlsx"

In [ ]:
# ## Open AI API
# os.environ["OPENAI_API_KEY"] = "your open ai api"
from dotenv import load_dotenv

load_dotenv()

## common functions

In [96]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [97]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()
        return content

In [98]:
"""
    This function is for the mathematical reasoning task.
"""
def construct_annotation(df, results, task_name, base_data_path, rule_id="0117"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema, presentation, calculation, definition, label, instance, us_gaap, true_answer, segmentation_path, parse_ok, input_col, outputByLLM = [[] for _ in range(12)]

    for res in results:
        folder_name = res.get("folder_name")
        prefix_path = os.path.join(base_data_path, folder_name)
        
        parse = False
        cal, pre, defn, lab, ins, sch, seg_path = [None] * 7
        
        ## special for mathematical reasoning task (DQC.US.0117)
        input_data = res.get("input", {})
        target_concept = input_data.get("target_concept")
        period = input_data.get("period")
        
        flattened_answers = [target_concept] if target_concept else []
        answers = res.get("output") 

        if os.path.exists(prefix_path) and os.path.isdir(prefix_path):
            all_subfolders = [d for d in os.listdir(prefix_path) if os.path.isdir(os.path.join(prefix_path, d))]

            for subfolder_name in all_subfolders:
                full_path = os.path.join(prefix_path, subfolder_name)
                temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch = [None] * 6
                
                file_list = os.listdir(full_path)
                for file_name in file_list:
                    final_path = os.path.join(full_path, file_name)
                    if file_name.endswith("cal.xml"): temp_cal = read_xml(final_path)
                    elif file_name.endswith("pre.xml"): temp_pre = read_xml(final_path)
                    elif file_name.endswith("def.xml"): temp_defn = read_xml(final_path)
                    elif file_name.endswith("lab.xml"): temp_lab = read_xml(final_path)
                    elif file_name.endswith("htm.xml"): temp_ins = read_xml(final_path)
                    elif file_name.endswith("xsd"): sch = read_xml(final_path)
                
                is_appear = True
                ## check the instance document
                instance_content = temp_ins or ""
                
                if not flattened_answers or not instance_content:
                    is_appear = False
                else:
                    for ans in flattened_answers:
                        if not (ans in instance_content or ans.replace(":", "_") in instance_content):
                            is_appear = False
                            break
               
                if is_appear:
                    parse = True
                    cal, pre, defn, lab, ins, sch = temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch
                    seg_path = os.path.join(folder_name, subfolder_name)
                    break 
        
        formatted_input = f'target concept: \"{target_concept}\"\nperiod: \"{period}\"' if target_concept else None
        
        calculation.append(cal); presentation.append(pre); definition.append(defn); label.append(lab); instance.append(ins); schema.append(sch);
        true_answer.append(answers); us_gaap.append(None); parse_ok.append(parse); segmentation_path.append(seg_path); input_col.append(formatted_input); outputByLLM.append(res);
     
    new_df["schema"] = schema; new_df["presentation"] = presentation; new_df["calculation"] = calculation; new_df["definition"] = definition;
    new_df["label"] = label; new_df["instance"] = instance; new_df["us_gaap"] = us_gaap; new_df["input"] = input_col;
    new_df["true_answer"] = true_answer; new_df["segmentation_path"] = segmentation_path; new_df["parse_successful"] = parse_ok; new_df["LLM_Output"] = outputByLLM;

    return new_df

## load dqc messages for mathematical reasoning task (DQC.US.0117)

In [99]:
mr_117_df = pd.read_csv(input_data)

In [100]:
mr_117_df.head()

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0117.9574,"2022-02-07 17:45:51,654 [DQC.US.0117.9574] In ...",2022-02-07,ERROR,DQC,117,913760,StoneX Group Inc.,2022-02-07 17:24:00,0000913760-22-000029,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-Q,http://www.sec.gov/Archives/edgar/data/913760/...,2022-02-07,2022,416718,https://www.sec.gov/Archives/edgar/data/913760...,6200,10q-intl-20211231
1,DQC.US.0117.9574,"2022-02-07 17:45:51,654 [DQC.US.0117.9574] In ...",2022-02-07,ERROR,DQC,117,913760,StoneX Group Inc.,2022-02-07 17:24:00,0000913760-22-000029,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-Q,http://www.sec.gov/Archives/edgar/data/913760/...,2022-02-07,2022,416718,https://www.sec.gov/Archives/edgar/data/913760...,6200,10q-intl-20211231
2,DQC.US.0117.9574,"2022-02-15 18:16:50,994 [DQC.US.0117.9574] In ...",2022-02-15,ERROR,DQC,117,89089,Service Corporation International,2022-02-15 17:23:00,0000089089-22-000012,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/89089/0...,2022-02-15,2022,420057,https://www.sec.gov/Archives/edgar/data/89089/...,7200,10k-sci-20211231
3,DQC.US.0117.9574,"2022-02-17 07:02:22,179 [DQC.US.0117.9574] In ...",2022-02-17,ERROR,DQC,117,92122,The Southern Company,2022-02-16 17:44:00,0000092122-22-000003,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/92122/0...,2022-02-17,2022,420498,https://www.sec.gov/Archives/edgar/data/92122/...,4911,10k-so-20211231
4,DQC.US.0117.9574,"2022-02-17 07:02:22,179 [DQC.US.0117.9574] In ...",2022-02-17,ERROR,DQC,117,92122,The Southern Company,2022-02-16 17:44:00,0000092122-22-000003,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/92122/0...,2022-02-17,2022,420498,https://www.sec.gov/Archives/edgar/data/92122/...,4911,10k-so-20211231


## LLM judge assisstant

In [ ]:
Client = OpenAI()

In [102]:
mr_117_system_prompt = """You are an expert financial analyst parsing DQC validation messages for mathematical reasoning errors (Rule DQC.US.0117). Your task is to extract all required components from the message.

1. **Statement Name**: Extract the clean name of the financial statement (e.g., "Condensed Consolidated Income Statements").

2. **Input Data**: From the properties list at the end of the message, extract the following:
    * `target_concept`: The full name of the primary `us-gaap:` concept being evaluated.
    * `period`: The reporting period for the fact (e.g., "2020-10-01 to 2020-12-31").

3. **Output Data**: From the main body of the message, extract the following values:
    * `extracted_value`: The value of the concept as reported in the filing (e.g., "255,500,000").
    * `calculated_value`: The correct value based on the dimensional breakdown (e.g., "142,400,000").
    * `is_correct`: This should always be "No" for these error messages.

Output your result as a single, structured JSON object in the following format:
```json
{
  "statement_name": "...",
  "input": {
    "target_concept": "us-gaap:RevenueFromContractWithCustomerIncludingAssessedTax",
    "period": "..."
  },
  "output": {
    "extracted_value": "...",
    "calculated_value": "...",
    "is_correct": "No"
  }
}
```"""

In [103]:
def get_response(dqc_message):
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": mr_117_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \\n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [104]:
res = []
# to run on the full dataset, use the line below.
for _, row in tqdm(mr_117_df.iterrows(), total=mr_117_df.shape[0]):
# to test on only the first 10 examples, comment out the line above and uncomment the line below:
# for _, row in tqdm(mr_117_df.head(10).iterrows(), total=10):
    
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    folder_name = row["folder_name"]
    segmentation_name = to_pascal_case(response.get("statement_name", ""))

    response["folder_name"] = folder_name
    response["segmentation_name"] = segmentation_name
    res.append(response)

100%|██████████| 120/120 [04:25<00:00,  2.21s/it]


In [105]:
res[0]

{'statement_name': 'Condensed Consolidated Income Statements',
 'input': {'target_concept': 'us-gaap:RevenueFromContractWithCustomerIncludingAssessedTax',
  'period': '2020-10-01 to 2020-12-31'},
 'output': {'extracted_value': '255,500,000',
  'calculated_value': '142,400,000',
  'is_correct': 'No'},
 'folder_name': '10q-intl-20211231',
 'segmentation_name': 'CondensedConsolidatedIncomeStatements'}

In [106]:
DATA_FOLDER_PATH = "./segmentation_data/mathematical_reasoning"

new_mr_117_df = construct_annotation(mr_117_df, res, 
                                     task_name="mathematical_reasoning", 
                                     base_data_path=DATA_FOLDER_PATH)

In [107]:
new_mr_117_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0117.9574,"2022-02-07 17:45:51,654 [DQC.US.0117.9574] In ...",2022-02-07,ERROR,DQC,117,913760,StoneX Group Inc.,2022-02-07 17:24:00,0000913760-22-000029,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"target concept: ""us-gaap:RevenueFromContractWi...","{'extracted_value': '255,500,000', 'calculated...",10q-intl-20211231/CondensedConsolidatedIncomeS...,True,{'statement_name': 'Condensed Consolidated Inc...
1,DQC.US.0117.9574,"2022-02-07 17:45:51,654 [DQC.US.0117.9574] In ...",2022-02-07,ERROR,DQC,117,913760,StoneX Group Inc.,2022-02-07 17:24:00,0000913760-22-000029,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"target concept: ""us-gaap:RevenueFromContractWi...","{'extracted_value': '920,700,000', 'calculated...",10q-intl-20211231/CondensedConsolidatedIncomeS...,True,{'statement_name': 'Condensed Consolidated Inc...
2,DQC.US.0117.9574,"2022-02-15 18:16:50,994 [DQC.US.0117.9574] In ...",2022-02-15,ERROR,DQC,117,89089,Service Corporation International,2022-02-15 17:23:00,0000089089-22-000012,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"target concept: ""us-gaap:AllowanceForDoubtfulA...","{'extracted_value': '19,204,000', 'calculated_...",10k-sci-20211231/PreneedActivitiesLongtermRece...,True,{'statement_name': 'Preneed Activities Receiva...
3,DQC.US.0117.9574,"2022-02-17 07:02:22,179 [DQC.US.0117.9574] In ...",2022-02-17,ERROR,DQC,117,92122,The Southern Company,2022-02-16 17:44:00,0000092122-22-000003,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"target concept: ""us-gaap:Revenues""\nperiod: ""2...","{'extracted_value': '21,419,000,000', 'calcula...",10k-so-20211231/StatementsofIncomeGPC,True,{'statement_name': 'Statements of Income - APC...
4,DQC.US.0117.9574,"2022-02-17 07:02:22,179 [DQC.US.0117.9574] In ...",2022-02-17,ERROR,DQC,117,92122,The Southern Company,2022-02-16 17:44:00,0000092122-22-000003,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"target concept: ""us-gaap:Revenues""\nperiod: ""2...","{'extracted_value': '21,419,000,000', 'calcula...",10k-so-20211231/StatementsofIncomeGPC,True,{'statement_name': 'Statements of Income - GPC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,DQC.US.0117.9574,"2024-11-19 13:17:18,105 [DQC.US.0117.9574] In ...",2024-11-19,ERROR,DQC,117,1563568,"ENVIROTECH VEHICLES, INC.",2024-11-14 16:15:00,0001437749-24-035310,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,"target concept: ""us-gaap:LongTermDebt""\nperiod...","{'extracted_value': '918,063', 'calculated_val...",10q-evtv-20240930/statement-consolidated-balan...,True,{'statement_name': '001 - Statement - Consolid...
116,DQC.US.0117.9574,"2024-11-21 06:12:15,663 [DQC.US.0117.9574] In ...",2024-11-21,ERROR,DQC,117,860543,Jacksam Corporation,2024-

In [108]:
new_mr_117_df.to_excel(output_data, index=False)